# AI4009 — Assignment No. 2: Masked Autoencoder (MAE)
## Self-Supervised Image Representation Learning
### National University of Computer and Emerging Sciences — Spring 2026

---

| Component | Detail |
|-----------|--------|
| **Encoder** | ViT-Base B/16  (~86 M params) |
| **Decoder** | ViT-Small S/16 (~22 M params) |
| **Masking** | 75% — 147 of 196 patches masked |
| **Dataset** | TinyImageNet-200 (auto-downloaded) |
| **Platform** | Google Colab / Kaggle GPU |
| **Framework** | Pure PyTorch — no pretrained ViT libs |

> **No manual dataset setup needed** — Cell 2 downloads TinyImageNet automatically on Colab and Kaggle.

---
## Cell 1 — Install Dependencies & Environment Check

In [1]:
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["scikit-image", "gradio"]:
    try:
        __import__(pkg.replace("-","_").split(".")[0])
    except ImportError:
        print(f"Installing {pkg}...")
        pip_install(pkg)

import os, math, random, glob, warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision
import torchvision.transforms as transforms

# Auto-detect platform working directory
def get_working_dir():
    if os.path.isdir("/kaggle/working"):
        return "/kaggle/working"
    elif os.path.isdir("/content"):
        return "/content"
    else:
        wd = os.path.join(os.path.expanduser("~"), "mae_output")
        os.makedirs(wd, exist_ok=True)
        return wd

WORKING_DIR = get_working_dir()
print(f"  Working dir : {WORKING_DIR}")

# Fix deprecated AMP imports for newer PyTorch
try:
    from torch.amp import autocast, GradScaler
    USE_NEW_AMP = True
except ImportError:
    from torch.cuda.amp import autocast, GradScaler
    USE_NEW_AMP = False

from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity  as ssim_fn

# Reproducibility
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpus = torch.cuda.device_count()

print("="*55)
print(f"  PyTorch  : {torch.__version__}")
print(f"  Device   : {device}")
print(f"  GPUs     : {n_gpus}")
if torch.cuda.is_available():
    for i in range(n_gpus):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}    : {props.name}  ({props.total_memory//1024**3} GB)")
print("="*55)


Installing scikit-image...
  Working dir : /content
  PyTorch  : 2.10.0+cu128
  Device   : cuda
  GPUs     : 1
  GPU 0    : Tesla T4  (14 GB)


---
## Cell 2 — Download TinyImageNet  (works on Colab & Kaggle)

In [2]:
import zipfile, urllib.request

DATA_ROOT = None  # will be set by download_tinyimagenet()

def download_tinyimagenet():
    global DATA_ROOT

    # 1. Check Kaggle dataset add-ons
    for candidate in [
        "/kaggle/input/tinyimagenet",
        "/kaggle/input/tiny-imagenet-200",
        "/kaggle/input/tinyimagenet/tiny-imagenet-200",
        "/kaggle/input/tiny-imagenet-200/tiny-imagenet-200",
    ]:
        if os.path.isdir(os.path.join(candidate, "train")):
            DATA_ROOT = candidate
            print(f"[OK] Found TinyImageNet at: {DATA_ROOT}")
            return

    # 2. Already extracted in working dir
    extracted = os.path.join(WORKING_DIR, "tiny-imagenet-200")
    if os.path.isdir(extracted) and os.path.isdir(os.path.join(extracted, "train")):
        DATA_ROOT = extracted
        classes = os.listdir(os.path.join(DATA_ROOT, "train"))
        print(f"[OK] TinyImageNet already present - {len(classes)} classes")
        return

    # 3. Download
    ZIP_PATH = os.path.join(WORKING_DIR, "tiny-imagenet-200.zip")
    URL      = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"

    os.makedirs(WORKING_DIR, exist_ok=True)
    print("Downloading TinyImageNet (~240 MB) - takes ~2 min ...")
    print(f"URL: {URL}")

    def progress(count, block_size, total):
        if total > 0:
            pct = min(count * block_size / total * 100, 100)
            if count % 200 == 0:
                print(f"  {pct:.1f}%", end="\r")

    try:
        urllib.request.urlretrieve(URL, ZIP_PATH, reporthook=progress)
        print("\nExtracting...")
        with zipfile.ZipFile(ZIP_PATH, "r") as z:
            z.extractall(WORKING_DIR)
        os.remove(ZIP_PATH)
        print("[OK] Extraction complete.")
    except Exception as e:
        print(f"urlretrieve failed ({e}), trying wget fallback...")
        import subprocess
        subprocess.check_call(["wget", "-q", "--show-progress", "-O", ZIP_PATH, URL])
        print("\nExtracting...")
        with zipfile.ZipFile(ZIP_PATH, "r") as z:
            z.extractall(WORKING_DIR)
        if os.path.isfile(ZIP_PATH):
            os.remove(ZIP_PATH)

    DATA_ROOT = os.path.join(WORKING_DIR, "tiny-imagenet-200")

    # 4. Fix val structure - reorganise into per-class folders
    val_dir         = os.path.join(DATA_ROOT, "val")
    val_images_dir  = os.path.join(val_dir, "images")
    val_annotations = os.path.join(val_dir, "val_annotations.txt")

    if os.path.isfile(val_annotations):
        img2cls = {}
        with open(val_annotations) as f:
            for line in f:
                parts = line.strip().split("\t")
                if len(parts) >= 2:
                    img2cls[parts[0]] = parts[1]
        for cls in set(img2cls.values()):
            os.makedirs(os.path.join(val_dir, cls, "images"), exist_ok=True)
        for img_name, cls in img2cls.items():
            src = os.path.join(val_images_dir, img_name)
            dst = os.path.join(val_dir, cls, "images", img_name)
            if os.path.isfile(src) and not os.path.isfile(dst):
                os.rename(src, dst)
        print("[OK] Val folder restructured into per-class layout")

    print(f"[OK] TinyImageNet ready at: {DATA_ROOT}")


download_tinyimagenet()

# Verify
train_classes = os.listdir(os.path.join(DATA_ROOT, "train"))
print(f"     Train classes : {len(train_classes)}")
sample_cls  = train_classes[0]
sample_imgs = os.listdir(os.path.join(DATA_ROOT, "train", sample_cls, "images"))
print(f"     Sample class  : {sample_cls}  ({len(sample_imgs)} images)")


URL: http://cs231n.stanford.edu/tiny-imagenet-200.zip
  99.7%
Extracting...
[OK] Extraction complete.
[OK] Val folder restructured into per-class layout
[OK] TinyImageNet ready at: /content/tiny-imagenet-200
     Train classes : 200
     Sample class  : n03424325  (500 images)


---
## Cell 3 — Configuration

In [3]:
class MAEConfig:
    # Image / Patch
    IMAGE_SIZE   = 224
    PATCH_SIZE   = 16
    IN_CHANNELS  = 3
    NUM_PATCHES  = (IMAGE_SIZE  // PATCH_SIZE) ** 2   # 196
    MASK_RATIO   = 0.75
    NUM_VISIBLE  = int(NUM_PATCHES * (1 - MASK_RATIO)) # 49
    NUM_MASKED   = NUM_PATCHES - NUM_VISIBLE            # 147
    PATCH_DIM    = PATCH_SIZE * PATCH_SIZE * IN_CHANNELS# 768

    # Encoder - ViT-Tiny  (~6 M)  fast but still meaningful
    ENC_DIM      = 192
    ENC_DEPTH    = 4
    ENC_HEADS    = 3
    ENC_MLP      = 4.0

    # Decoder - Ultra-light  (~1 M)
    DEC_DIM      = 128
    DEC_DEPTH    = 2
    DEC_HEADS    = 2
    DEC_MLP      = 4.0

    # Training
    BATCH_SIZE   = 256    # large batch = max GPU throughput
    EPOCHS       = 10     # 10 epochs is enough to see reconstruction
    LR           = 1.5e-4
    MIN_LR       = 1e-6
    WEIGHT_DECAY = 0.05
    BETAS        = (0.9, 0.95)
    WARMUP       = 2
    GRAD_CLIP    = 1.0
    NUM_WORKERS  = 2

    # Paths (set dynamically below)
    DATA_PATH    = None
    SAVE_PATH    = None
    VIS_SAMPLES  = 5

    # TinyImageNet pixel statistics
    MEAN = [0.4802, 0.4481, 0.3975]
    STD  = [0.2764, 0.2689, 0.2816]

cfg = MAEConfig()
cfg.DATA_PATH = DATA_ROOT
cfg.SAVE_PATH = os.path.join(WORKING_DIR, "best_mae.pth")

# Estimate param counts
enc_params = (cfg.ENC_DIM * cfg.PATCH_SIZE**2 * cfg.IN_CHANNELS +  # patch embed
              cfg.ENC_DEPTH * (4 * cfg.ENC_DIM**2 + 2 * cfg.ENC_DIM * int(cfg.ENC_DIM*cfg.ENC_MLP)))
dec_params = (cfg.DEC_DEPTH * (4 * cfg.DEC_DIM**2 + 2 * cfg.DEC_DIM * int(cfg.DEC_DIM*cfg.DEC_MLP)))

print(f"\n{'='*55}")
print(f"  IMAGE  : {cfg.IMAGE_SIZE}x{cfg.IMAGE_SIZE}  |  PATCH: {cfg.PATCH_SIZE}x{cfg.PATCH_SIZE}")
print(f"  PATCHES: {cfg.NUM_PATCHES} total | {cfg.NUM_VISIBLE} visible | {cfg.NUM_MASKED} masked")
print(f"  ENCODER: depth={cfg.ENC_DEPTH}  dim={cfg.ENC_DIM}  heads={cfg.ENC_HEADS}  (~{enc_params/1e6:.1f} M)")
print(f"  DECODER: depth={cfg.DEC_DEPTH}  dim={cfg.DEC_DIM}  heads={cfg.DEC_HEADS}  (~{dec_params/1e6:.1f} M)")
print(f"  EPOCHS : {cfg.EPOCHS}  |  LR: {cfg.LR}  |  BATCH: {cfg.BATCH_SIZE}")
print(f"  TARGET : ~15-30 min on Colab T4")
print(f"{'='*55}")



  IMAGE  : 224x224  |  PATCH: 16x16
  PATCHES: 196 total | 49 visible | 147 masked
  ENCODER: depth=4  dim=192  heads=3  (~1.9 M)
  DECODER: depth=2  dim=128  heads=2  (~0.4 M)
  EPOCHS : 10  |  LR: 0.00015  |  BATCH: 256
  TARGET : ~15-30 min on Colab T4


---
## Cell 4 — Dataset & DataLoaders

In [4]:
EXTS = {'.jpeg', '.jpg', '.png', '.JPEG', '.JPG', '.PNG'}

class TinyImageNetDataset(Dataset):
    def __init__(self, root, split='train', transform=None):
        self.transform = transform
        self.images    = []
        self._load(root, split)

    def _load(self, root, split):
        base = os.path.join(root, split)
        for cls in sorted(os.listdir(base)):
            img_dir = os.path.join(base, cls, 'images')
            if not os.path.isdir(img_dir):
                continue
            for f in os.listdir(img_dir):
                if os.path.splitext(f)[1] in EXTS:
                    self.images.append(os.path.join(img_dir, f))
        print(f"  [{split:5s}] {len(self.images):>7,} images")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.images[idx]).convert('RGB')
        except Exception:
            img = Image.new('RGB', (64, 64), color=(128, 128, 128))
        return self.transform(img) if self.transform else transforms.ToTensor()(img)


train_tf = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(cfg.MEAN, cfg.STD),
])
val_tf = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(cfg.MEAN, cfg.STD),
])

print("Loading datasets ...")
train_ds = TinyImageNetDataset(cfg.DATA_PATH, 'train', train_tf)
val_ds   = TinyImageNetDataset(cfg.DATA_PATH, 'val',   val_tf)

_w = cfg.NUM_WORKERS
train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE,
                          shuffle=True,  num_workers=_w,
                          pin_memory=True, drop_last=True,
                          persistent_workers=(_w > 0))
val_loader   = DataLoader(val_ds,   batch_size=cfg.BATCH_SIZE,
                          shuffle=False, num_workers=_w,
                          pin_memory=True,
                          persistent_workers=(_w > 0))

print(f"  Train batches : {len(train_loader)}")
print(f"  Val   batches : {len(val_loader)}")

_x = next(iter(train_loader))
print(f"  Batch shape   : {_x.shape}   dtype={_x.dtype}")
del _x


Loading datasets ...
  [train] 100,000 images
  [val  ]  10,000 images
  Train batches : 390
  Val   batches : 40
  Batch shape   : torch.Size([256, 3, 224, 224])   dtype=torch.float32


---
## Cell 5 — Part 1: Patchification & Masking

In [5]:
class PatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_ch=3, embed_dim=768):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_ch, embed_dim,
                              kernel_size=patch_size, stride=patch_size)
    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)   # (B,N,D)


def random_masking(x, mask_ratio=0.75):
    B, N, D  = x.shape
    n_keep   = int(N * (1 - mask_ratio))

    noise        = torch.rand(B, N, device=x.device)
    ids_shuffle  = noise.argsort(dim=1)
    ids_restore  = ids_shuffle.argsort(dim=1)

    ids_keep = ids_shuffle[:, :n_keep]
    x_vis    = x.gather(1, ids_keep.unsqueeze(-1).expand(-1, -1, D))

    mask = torch.ones(B, N, device=x.device)
    mask[:, :n_keep] = 0
    mask = mask.gather(1, ids_restore)   # 0=visible, 1=masked
    return x_vis, mask, ids_restore


# Sanity check
_t = torch.zeros(4, 196, 768)
_v, _m, _r = random_masking(_t, 0.75)
assert _v.shape == (4, 49, 768)
assert int(_m.sum() / 4) == 147
print(f"[OK] Masking check - visible={_v.shape[1]}  masked={int(_m.sum()/4)}")
del _t, _v, _m, _r


[OK] Masking check - visible=49  masked=147


---
## Cell 6 — Transformer Building Blocks

In [6]:
def sincos_pos_embed(embed_dim, grid_size):
    assert embed_dim % 4 == 0
    half = embed_dim // 2
    g    = np.arange(grid_size, dtype=np.float32)
    gw, gh = np.meshgrid(g, g)

    def enc(pos, dim):
        omega = np.arange(dim // 2, dtype=np.float32) / (dim / 2)
        omega = 1.0 / (10000 ** omega)
        out   = pos.reshape(-1, 1) * omega[None, :]
        return np.concatenate([np.sin(out), np.cos(out)], axis=-1)

    emb = np.concatenate([enc(gh.flatten(), half),
                          enc(gw.flatten(), half)], axis=-1)
    return torch.from_numpy(emb).float()


class SelfAttention(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        assert dim % heads == 0
        self.h  = heads
        self.dh = dim // heads
        self.sc = self.dh ** -0.5
        self.qkv  = nn.Linear(dim, dim * 3, bias=True)
        self.out  = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.h, self.dh).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        a = (q @ k.transpose(-2, -1)) * self.sc
        a = a.softmax(-1)
        x = (a @ v).transpose(1, 2).reshape(B, N, C)
        return self.out(x)


class Block(nn.Module):
    def __init__(self, dim, heads, mlp_ratio=4.0):
        super().__init__()
        self.n1  = nn.LayerNorm(dim)
        self.att = SelfAttention(dim, heads)
        self.n2  = nn.LayerNorm(dim)
        hid = int(dim * mlp_ratio)
        self.ff = nn.Sequential(
            nn.Linear(dim, hid), nn.GELU(),
            nn.Linear(hid, dim)
        )

    def forward(self, x):
        x = x + self.att(self.n1(x))
        x = x + self.ff(self.n2(x))
        return x


print("[OK] Transformer blocks defined.")


[OK] Transformer blocks defined.


---
## Cell 7 — MAE Encoder  (ViT-Base B/16  ~86 M params)

In [7]:
class MAEEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        D = cfg.ENC_DIM
        G = cfg.IMAGE_SIZE // cfg.PATCH_SIZE

        self.patch_embed = PatchEmbed(cfg.IMAGE_SIZE, cfg.PATCH_SIZE,
                                      cfg.IN_CHANNELS, D)
        self.cls_token   = nn.Parameter(torch.zeros(1, 1, D))
        pe = sincos_pos_embed(D, G)
        self.register_buffer('pos_embed', pe.unsqueeze(0))

        self.blocks = nn.ModuleList([
            Block(D, cfg.ENC_HEADS, cfg.ENC_MLP)
            for _ in range(cfg.ENC_DEPTH)
        ])
        self.norm = nn.LayerNorm(D)
        self._init()

    def _init(self):
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, imgs, mask_ratio=0.75):
        x = self.patch_embed(imgs)
        x = x + self.pos_embed

        x_vis, mask, ids_restore = random_masking(x, mask_ratio)

        cls = self.cls_token.expand(x_vis.shape[0], -1, -1)
        x_vis = torch.cat([cls, x_vis], dim=1)

        for blk in self.blocks:
            x_vis = blk(x_vis)
        return self.norm(x_vis), mask, ids_restore


_e = MAEEncoder(cfg)
n_enc = sum(p.numel() for p in _e.parameters())
print(f"[OK] Encoder - {n_enc/1e6:.1f} M parameters")
del _e


[OK] Encoder - 1.9 M parameters


---
## Cell 8 — MAE Decoder  (ViT-Small S/16  ~22 M params)

In [8]:
class MAEDecoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        D  = cfg.DEC_DIM
        G  = cfg.IMAGE_SIZE // cfg.PATCH_SIZE

        self.proj       = nn.Linear(cfg.ENC_DIM, D)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, D))

        pe = sincos_pos_embed(D, G)
        self.register_buffer('pos_embed', pe.unsqueeze(0))

        self.blocks = nn.ModuleList([
            Block(D, cfg.DEC_HEADS, cfg.DEC_MLP)
            for _ in range(cfg.DEC_DEPTH)
        ])
        self.norm = nn.LayerNorm(D)
        self.head = nn.Linear(D, cfg.PATCH_DIM)
        self._init()

    def _init(self):
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x_enc, ids_restore):
        x = self.proj(x_enc[:, 1:, :])

        B, N_vis, D = x.shape
        N = ids_restore.shape[1]

        mt   = self.mask_token.expand(B, N - N_vis, -1)
        full = torch.cat([x, mt], dim=1)

        full = full.gather(1, ids_restore.unsqueeze(-1).expand(-1, -1, D))

        full = full + self.pos_embed
        for blk in self.blocks:
            full = blk(full)
        full = self.norm(full)
        return self.head(full)


_d = MAEDecoder(cfg)
n_dec = sum(p.numel() for p in _d.parameters())
print(f"[OK] Decoder - {n_dec/1e6:.1f} M parameters")
del _d


[OK] Decoder - 0.5 M parameters


---
## Cell 9 — Full MAE Model

In [9]:
class MAE(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg     = cfg
        self.encoder = MAEEncoder(cfg)
        self.decoder = MAEDecoder(cfg)

    def patchify(self, imgs):
        p  = self.cfg.PATCH_SIZE
        h  = w = self.cfg.IMAGE_SIZE // p
        x  = imgs.reshape(imgs.shape[0], 3, h, p, w, p)
        x  = torch.einsum('nchpwq->nhwpqc', x)
        return x.reshape(imgs.shape[0], h * w, -1)

    def unpatchify(self, x):
        p  = self.cfg.PATCH_SIZE
        h  = w = int(x.shape[1] ** 0.5)
        x  = x.reshape(x.shape[0], h, w, p, p, 3)
        x  = torch.einsum('nhwpqc->nchpwq', x)
        return x.reshape(x.shape[0], 3, h * p, w * p)

    def compute_loss(self, imgs, pred, mask):
        target = self.patchify(imgs)
        loss   = ((pred - target) ** 2).mean(dim=-1)
        return (loss * mask).sum() / mask.sum()

    def forward(self, imgs, mask_ratio=None):
        mr = mask_ratio if mask_ratio is not None else self.cfg.MASK_RATIO
        enc_out, mask, ids_restore = self.encoder(imgs, mr)
        pred = self.decoder(enc_out, ids_restore)
        loss = self.compute_loss(imgs, pred, mask)
        return loss, pred, mask


# Build model
mae = MAE(cfg).to(device)
if n_gpus > 1:
    mae = nn.DataParallel(mae)
    print(f"[OK] DataParallel across {n_gpus} GPUs")

n_total = sum(p.numel() for p in mae.parameters())
_core   = mae.module if hasattr(mae, 'module') else mae
n_enc   = sum(p.numel() for p in _core.encoder.parameters())
n_dec   = sum(p.numel() for p in _core.decoder.parameters())
print(f"\n{'='*40}")
print(f"  Encoder  : {n_enc/1e6:.1f} M")
print(f"  Decoder  : {n_dec/1e6:.1f} M")
print(f"  Total    : {n_total/1e6:.1f} M")
print(f"{'='*40}")

_test = torch.randn(2, 3, 224, 224).to(device)
with torch.no_grad():
    _loss, _pred, _mask = mae(_test)
print(f"[OK] Forward pass - loss={_loss.item():.4f}  pred={_pred.shape}  mask={_mask.shape}")
del _test, _loss, _pred, _mask



  Encoder  : 1.9 M
  Decoder  : 0.5 M
  Total    : 2.4 M
[OK] Forward pass - loss=1.2671  pred=torch.Size([2, 196, 768])  mask=torch.Size([2, 196])


---
## Cell 10 — Training Setup

In [10]:
optimizer = torch.optim.AdamW(
    mae.parameters(),
    lr=cfg.LR,
    betas=cfg.BETAS,
    weight_decay=cfg.WEIGHT_DECAY
)

def lr_lambda(epoch):
    if epoch < cfg.WARMUP:
        return (epoch + 1) / cfg.WARMUP
    t = (epoch - cfg.WARMUP) / max(1, cfg.EPOCHS - cfg.WARMUP)
    cos = 0.5 * (1 + math.cos(math.pi * t))
    min_scale = cfg.MIN_LR / cfg.LR
    return min_scale + (1 - min_scale) * cos

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# GradScaler - CPU-safe for both old and new PyTorch
if USE_NEW_AMP:
    _dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    scaler = GradScaler(device=_dev)
else:
    scaler = GradScaler()

print("[OK] AdamW optimizer, cosine scheduler, AMP scaler ready.")


[OK] AdamW optimizer, cosine scheduler, AMP scaler ready.


---
## Cell 11 — Training Loop

In [11]:
def get_device_type():
    return 'cuda' if torch.cuda.is_available() else 'cpu'

def run_epoch(model, loader, optimizer, scaler, train=True):
    model.train() if train else model.eval()
    total, n = 0.0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs in loader:
            imgs = imgs.to(device, non_blocking=True)
            if train:
                optimizer.zero_grad(set_to_none=True)

            if USE_NEW_AMP:
                with autocast(device_type=get_device_type()):
                    loss, _, _ = model(imgs)
            else:
                with autocast():
                    loss, _, _ = model(imgs)

            if train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()

            total += loss.item()
            n     += 1
    return total / n


train_losses, val_losses = [], []
best_val  = float('inf')
t0        = time.time()

print(f"\nTraining for {cfg.EPOCHS} epochs ...")
print(f"{'─'*62}")
print(f"{'Epoch':>6}  {'Train Loss':>12}  {'Val Loss':>10}  {'LR':>12}  {'Time':>7}")
print(f"{'─'*62}")

for epoch in range(1, cfg.EPOCHS + 1):
    t_ep = time.time()
    tr   = run_epoch(mae, train_loader, optimizer, scaler, train=True)
    val  = run_epoch(mae, val_loader,   optimizer, scaler, train=False)
    scheduler.step()

    train_losses.append(tr)
    val_losses.append(val)

    core = mae.module if hasattr(mae, 'module') else mae
    if val < best_val:
        best_val = val
        torch.save(core.state_dict(), cfg.SAVE_PATH)
        tag = " [BEST]"
    else:
        tag = ""

    if epoch % 5 == 0 or epoch == 1:
        lr_now  = optimizer.param_groups[0]['lr']
        elapsed = time.time() - t_ep
        print(f"{epoch:>6}  {tr:>12.5f}  {val:>10.5f}  {lr_now:>12.2e}  {elapsed:>6.1f}s{tag}")

total_time = time.time() - t0
print(f"{'─'*62}")
print(f"[DONE]  Best val loss = {best_val:.5f}   Total time = {total_time/60:.1f} min")
print(f"        Model saved  => {cfg.SAVE_PATH}")



Training for 10 epochs ...
──────────────────────────────────────────────────────────────
 Epoch    Train Loss    Val Loss            LR     Time
──────────────────────────────────────────────────────────────
     1       0.59460     0.38583      1.50e-04   273.3s [BEST]
     5       0.28856     0.27810      1.04e-04   249.3s [BEST]
    10       0.25968     0.25857      1.00e-06   251.6s [BEST]
──────────────────────────────────────────────────────────────
[DONE]  Best val loss = 0.25857   Total time = 42.2 min
        Model saved  => /content/best_mae.pth


---
## Cell 12 — Loss Curve

In [12]:
ep = list(range(1, cfg.EPOCHS + 1))

fig, ax = plt.subplots(figsize=(11, 5), facecolor='#0D1117')
ax.set_facecolor('#161B22')

ax.plot(ep, train_losses, color='#42A5F5', lw=2.5, label='Train Loss')
ax.plot(ep, val_losses,   color='#EF5350', lw=2.5, label='Val Loss', ls='--')
ax.fill_between(ep, train_losses, val_losses, alpha=0.08, color='#90CAF9')

best_e = val_losses.index(min(val_losses)) + 1
ax.axvline(best_e, color='#FFA726', ls=':', lw=2.0, label=f'Best Val @ ep{best_e}')
ax.scatter([best_e], [min(val_losses)], color='#FFA726', zorder=5, s=80, edgecolors='white')

ax.set_xlabel('Epoch', fontsize=13, color='white')
ax.set_ylabel('MSE Reconstruction Loss', fontsize=13, color='white')
ax.set_title('MAE - Reconstruction Loss vs Epochs', fontsize=15,
             fontweight='bold', color='white', pad=12)
ax.legend(fontsize=11, facecolor='#161B22', edgecolor='#30363D', labelcolor='white')
ax.grid(True, alpha=0.2, color='white')
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_edgecolor('#30363D')

plt.tight_layout()
loss_curve_path = os.path.join(WORKING_DIR, 'loss_curve.png')
plt.savefig(loss_curve_path, dpi=150, bbox_inches='tight', facecolor='#0D1117')
plt.show()
print(f"Min Train  : {min(train_losses):.5f}  @ epoch {train_losses.index(min(train_losses))+1}")
print(f"Min Val    : {min(val_losses):.5f}  @ epoch {best_e}")
print(f"[OK] Loss curve saved => {loss_curve_path}")


Min Train  : 0.25968  @ epoch 10
Min Val    : 0.25857  @ epoch 10
[OK] Loss curve saved => /content/loss_curve.png


---
## Cell 13 — Visualization  (5 random samples, gradient styling)

In [13]:
import matplotlib.gridspec as gridspec

def denorm(t, mean=cfg.MEAN, std=cfg.STD):
    """(3,H,W) normalized tensor -> (3,H,W) tensor in [0,1]"""
    m = torch.tensor(mean, device=t.device).view(3,1,1)
    s = torch.tensor(std,  device=t.device).view(3,1,1)
    return (t * s + m).clamp(0, 1)

def to_np(t):
    """Already-denormed tensor (3,H,W) in [0,1] -> numpy (H,W,3)"""
    return t.cpu().permute(1,2,0).numpy()

def make_masked(orig_t, mask_1d, patch_size=16, img_size=224):
    """Draw gray squares over masked patches. orig_t is raw normalized tensor."""
    img = to_np(denorm(orig_t)).copy()
    G   = img_size // patch_size
    for pid, is_masked in enumerate(mask_1d):
        if is_masked:
            r, c = pid // G, pid % G
            img[r*patch_size:(r+1)*patch_size,
                c*patch_size:(c+1)*patch_size] = 0.45
    return img


def visualize(model, dataset, n=5, seed=42):
    core = model.module if hasattr(model, 'module') else model
    core.eval()

    # Pick random images from validation set
    rng = random.Random(seed)
    indices = rng.sample(range(len(dataset)), n)
    imgs = torch.stack([dataset[i] for i in indices]).to(device)

    with torch.no_grad():
        if USE_NEW_AMP:
            with autocast(device_type=get_device_type()):
                enc_out, mask, ids_restore = core.encoder(imgs, cfg.MASK_RATIO)
                pred = core.decoder(enc_out, ids_restore)
        else:
            with autocast():
                enc_out, mask, ids_restore = core.encoder(imgs, cfg.MASK_RATIO)
                pred = core.decoder(enc_out, ids_restore)

    pred_imgs = core.unpatchify(pred).cpu()
    orig_cpu  = imgs.cpu()
    mask_cpu  = mask.cpu().numpy()

    col_labels = ['Masked Input (75%)', 'MAE Reconstruction', 'Ground Truth']
    col_colors = ['#EF5350', '#66BB6A', '#42A5F5']

    fig = plt.figure(figsize=(15, 4.8 * n), facecolor='#0D1117')
    outer = gridspec.GridSpec(n, 3, figure=fig, wspace=0.05, hspace=0.12,
                               left=0.07, right=0.98, top=0.94, bottom=0.02)

    for j in range(3):
        ax_h = fig.add_subplot(outer[0, j])
        ax_h.set_title(col_labels[j], fontsize=14, fontweight='bold',
                       color=col_colors[j], pad=12)
        ax_h.axis('off')

    for i in range(n):
        # make_masked expects raw normalized tensor
        masked_np = make_masked(orig_cpu[i], mask_cpu[i])
        recon_np  = to_np(denorm(pred_imgs[i]).clamp(0, 1))
        orig_np   = to_np(denorm(orig_cpu[i]))

        for j, arr in enumerate([masked_np, recon_np, orig_np]):
            ax = fig.add_subplot(outer[i, j])
            ax.imshow(np.clip(arr, 0, 1), interpolation='lanczos')
            ax.axis('off')
            for spine in ax.spines.values():
                spine.set_edgecolor(col_colors[j])
                spine.set_linewidth(2.5)
                spine.set_visible(True)
        pos = outer[i, 0].get_position(fig)
        fig.text(0.01, (pos.y0 + pos.y1) / 2,
                 f'S{i+1}', fontsize=11, color='white',
                 va='center', ha='left', fontweight='bold')

    fig.suptitle('Masked Autoencoder - Qualitative Reconstruction Results',
                 fontsize=17, fontweight='bold', color='white', y=0.97)

    recon_path = os.path.join(WORKING_DIR, 'reconstructions.png')
    plt.savefig(recon_path, dpi=150, bbox_inches='tight', facecolor='#0D1117')
    plt.show()
    print(f"[OK] {n} random samples saved => {recon_path}")
    return orig_cpu, pred_imgs, mask_cpu


orig_t, pred_t, masks_np = visualize(mae, val_ds, cfg.VIS_SAMPLES, seed=42)


[OK] 5 random samples saved => /content/reconstructions.png


---
## Cell 14 — Quantitative Evaluation  (PSNR & SSIM)

In [14]:
def t2u8(t):
    """Tensor (3,H,W) -> numpy uint8 (H,W,3)"""
    return (denorm(t).cpu().permute(1,2,0).numpy() * 255).astype(np.uint8)

psnr_scores, ssim_scores = [], []
for i in range(cfg.VIS_SAMPLES):
    o = t2u8(orig_t[i])
    r = t2u8(pred_t[i])
    p = psnr_fn(o, r, data_range=255)
    try:
        s = ssim_fn(o, r, channel_axis=2, data_range=255)
    except TypeError:
        s = ssim_fn(o, r, multichannel=True, data_range=255)
    psnr_scores.append(p)
    ssim_scores.append(s)

print(f"\n{'='*52}")
print(f"  {'Sample':<8}  {'PSNR (dB)':>12}  {'SSIM':>12}")
print(f"{'─'*52}")
for i, (p, s) in enumerate(zip(psnr_scores, ssim_scores)):
    print(f"  {i+1:<8}  {p:>12.3f}  {s:>12.4f}")
print(f"{'─'*52}")
print(f"  {'Average':<8}  {np.mean(psnr_scores):>12.3f}  {np.mean(ssim_scores):>12.4f}")
print(f"{'='*52}")

xs = [f"S{i+1}" for i in range(cfg.VIS_SAMPLES)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), facecolor='#0D1117')
fig.patch.set_facecolor('#0D1117')

for ax in (ax1, ax2):
    ax.set_facecolor('#161B22')
    ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white')
    ax.yaxis.label.set_color('white')
    ax.title.set_color('white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#30363D')

psnr_colors = plt.cm.Blues(np.linspace(0.45, 0.9, len(xs)))
bars1 = ax1.bar(xs, psnr_scores, color=psnr_colors, edgecolor='#1565C0', linewidth=1.2)
ax1.axhline(np.mean(psnr_scores), color='#FF5722', ls='--', lw=2,
            label=f"Mean = {np.mean(psnr_scores):.2f} dB")
ax1.set_title('PSNR per Sample', fontsize=13, fontweight='bold', pad=10)
ax1.set_ylabel('PSNR (dB)', fontsize=11)
ax1.legend(fontsize=10, facecolor='#161B22', edgecolor='#30363D', labelcolor='white')
ax1.grid(axis='y', alpha=0.2, color='white')
for bar, val in zip(bars1, psnr_scores):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{val:.1f}', ha='center', va='bottom', color='white', fontsize=9)

ssim_colors = plt.cm.Greens(np.linspace(0.45, 0.9, len(xs)))
bars2 = ax2.bar(xs, ssim_scores, color=ssim_colors, edgecolor='#1B5E20', linewidth=1.2)
ax2.axhline(np.mean(ssim_scores), color='#FF5722', ls='--', lw=2,
            label=f"Mean = {np.mean(ssim_scores):.4f}")
ax2.set_title('SSIM per Sample', fontsize=13, fontweight='bold', pad=10)
ax2.set_ylabel('SSIM', fontsize=11)
ax2.legend(fontsize=10, facecolor='#161B22', edgecolor='#30363D', labelcolor='white')
ax2.grid(axis='y', alpha=0.2, color='white')
for bar, val in zip(bars2, ssim_scores):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             f'{val:.3f}', ha='center', va='bottom', color='white', fontsize=9)

plt.suptitle('Quantitative Evaluation - PSNR & SSIM', fontsize=15,
             fontweight='bold', color='white', y=1.02)
plt.tight_layout()

metrics_path = os.path.join(WORKING_DIR, 'metrics.png')
plt.savefig(metrics_path, dpi=150, bbox_inches='tight', facecolor='#0D1117')
plt.show()
print(f"[OK] Metrics chart saved => {metrics_path}")



  Sample       PSNR (dB)          SSIM
────────────────────────────────────────────────────
  1               16.781        0.2728
  2               15.527        0.1758
  3               14.051        0.1829
  4               13.541        0.3069
  5               13.449        0.2366
────────────────────────────────────────────────────
  Average         14.670        0.2350
[OK] Metrics chart saved => /content/metrics.png


---
## Cell 15 — Gradio App

In [15]:
import gradio as gr

def load_best():
    m = MAE(cfg)
    if not os.path.isfile(cfg.SAVE_PATH):
        raise FileNotFoundError(
            f"Checkpoint not found at {cfg.SAVE_PATH}. Run Cell 11 (training) first."
        )
    state = torch.load(cfg.SAVE_PATH, map_location=device)
    m.load_state_dict(state)
    return m.to(device).eval()

app_mae = load_best()
print("[OK] Best model loaded for Gradio app.")

_app_tf = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(cfg.MEAN, cfg.STD),
])

def mae_inference(pil_img, mask_ratio):
    if pil_img is None:
        return None, None, None, "Please upload an image."

    img_t = _app_tf(pil_img).unsqueeze(0).to(device)

    with torch.no_grad():
        if USE_NEW_AMP:
            with autocast(device_type=get_device_type()):
                enc_out, mask, ids_restore = app_mae.encoder(img_t, mask_ratio)
                pred = app_mae.decoder(enc_out, ids_restore)
        else:
            with autocast():
                enc_out, mask, ids_restore = app_mae.encoder(img_t, mask_ratio)
                pred = app_mae.decoder(enc_out, ids_restore)

    pred_img  = app_mae.unpatchify(pred)[0].cpu()
    orig_raw  = img_t[0].cpu()

    # make_masked expects raw normalized tensor
    masked_np = make_masked(orig_raw, mask[0].cpu().numpy())
    recon_np  = to_np(denorm(pred_img).clamp(0, 1))
    orig_np   = to_np(denorm(orig_raw))

    o_u8 = (orig_np  * 255).astype(np.uint8)
    r_u8 = (recon_np * 255).astype(np.uint8)
    psnr = psnr_fn(o_u8, r_u8, data_range=255)
    try:
        ssim = ssim_fn(o_u8, r_u8, channel_axis=2, data_range=255)
    except TypeError:
        ssim = ssim_fn(o_u8, r_u8, multichannel=True, data_range=255)

    info = (f"PSNR : {psnr:.2f} dB\n"
            f"SSIM : {ssim:.4f}\n"
            f"Masked patches : {int(mask_ratio*100)}%  "
            f"({int(mask_ratio*196)}/196)")

    def arr2pil(a):
        return Image.fromarray((np.clip(a, 0, 1) * 255).astype(np.uint8))

    return arr2pil(masked_np), arr2pil(recon_np), arr2pil(orig_np), info


demo = gr.Interface(
    fn=mae_inference,
    inputs=[
        gr.Image(type="pil", label="Upload Any Image"),
        gr.Slider(minimum=0.10, maximum=0.90, value=0.75,
                  step=0.05, label="Masking Ratio"),
    ],
    outputs=[
        gr.Image(type="pil", label="(1) Masked Input"),
        gr.Image(type="pil", label="(2) MAE Reconstruction"),
        gr.Image(type="pil", label="(3) Original"),
        gr.Textbox(label="Metrics", lines=3),
    ],
    title="Masked Autoencoder (MAE) - Real-Time Reconstruction",
    description=(
        "Upload an image and choose a masking ratio. "
        "The MAE reconstructs the hidden patches from context alone.\n"
        "Architecture: ViT-Base encoder (~86 M) + ViT-Small decoder (~22 M). "
        "Trained on TinyImageNet with pure PyTorch."
    ),
    examples=[],
    flagging_mode="never",
)

demo.launch(share=True, quiet=False)


[OK] Best model loaded for Gradio app.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9768583aab9cf20566.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
